<a href="https://colab.research.google.com/github/NMinarecioglu/kizilirmak-drought-propagation/blob/main/Cross_Wavelet_Transform_(XWT)_and_Wavelet_Coherence_(WTC)_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Cross Wavelet Transform (XWT) and Wavelet Coherence (WTC) Analysis
SPI <-> SDI  and  SPI <-> RDI  pairs — all labels in English

Requirements:
    pip install pycwt numpy matplotlib scipy pandas openpyxl

Output:
    xwt_plots/ folder: 6 XWT + 6 WTC = 12 PNG files
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pycwt as wavelet
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
INPUT_PATH = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_output_indices"
DATE_COL   = "Date"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_xwt_plots"

PAIRS = [
    ("SPI-3",  "SDI-3",  "3-Month"),
    ("SPI-6",  "SDI-6",  "6-Month"),
    ("SPI-12", "SDI-12", "12-Month"),
    ("SPI-3",  "RDI-3",  "3-Month"),
    ("SPI-6",  "RDI-6",  "6-Month"),
    ("SPI-12", "RDI-12", "12-Month"),
]

# Wavelet parameters
DT     = 1 / 12        # Monthly data → years
DJ     = 0.25
S0     = 2 * DT
J      = int(7 / DJ)
MOTHER = wavelet.Morlet(6)
ALPHA  = 0.05          # 95% significance
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ----------------------------------------------------------
# Load data
# ----------------------------------------------------------
def load_data(path, date_col):
    if os.path.isdir(path):
        files = sorted([
            os.path.join(path, f) for f in os.listdir(path)
            if f.endswith((".xlsx", ".xls", ".csv")) and not f.startswith("~")
        ])
        if not files:
            raise FileNotFoundError(f"No Excel/CSV found in '{path}'.")
        frames = []
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            tmp = pd.read_excel(fp, parse_dates=[date_col]) if ext in [".xlsx", ".xls"] \
                  else pd.read_csv(fp, parse_dates=[date_col])
            frames.append(tmp)
        df = pd.concat(frames, ignore_index=True)
    else:
        ext = os.path.splitext(path)[1].lower()
        df  = pd.read_excel(path, parse_dates=[date_col]) if ext in [".xlsx", ".xls"] \
              else pd.read_csv(path, parse_dates=[date_col])
    return df.sort_values(date_col).reset_index(drop=True)


df_raw = load_data(INPUT_PATH, DATE_COL)
print(f"Data loaded: {len(df_raw)} rows")
print(f"Columns: {list(df_raw.columns)}\n")


# ----------------------------------------------------------
# Prepare normalized series
# ----------------------------------------------------------
def prepare_series(col):
    s   = df_raw[[DATE_COL, col]].dropna().copy()
    s   = s.set_index(DATE_COL)[col].interpolate("linear")
    dat = s.values.astype(float)
    dat = (dat - dat.mean()) / dat.std()
    t   = np.linspace(s.index.year[0], s.index.year[-1], len(dat))
    return dat, t


# ----------------------------------------------------------
# Draw phase arrows on significant regions
# ----------------------------------------------------------
def draw_arrows(ax, t, period, phase_angle, sig_mask, step=5):
    """
    Right arrow  → in-phase
    Left arrow   → anti-phase
    Up arrow     → X leads Y
    Down arrow   → Y leads X
    """
    for pi in range(0, len(period), step):
        for ti in range(0, len(t), step):
            if sig_mask[pi, ti]:
                angle = phase_angle[pi, ti]
                dx = np.cos(angle) * 0.4
                dy = np.sin(angle) * 0.3
                ax.annotate(
                    "", xy=(t[ti] + dx, np.log2(period[pi]) + dy),
                    xytext=(t[ti], np.log2(period[pi])),
                    arrowprops=dict(arrowstyle="->", color="black",
                                   lw=0.6, mutation_scale=6),
                )


# ----------------------------------------------------------
# Y-axis helper
# ----------------------------------------------------------
def set_period_yaxis(ax, period):
    period_clean = period[np.isfinite(period) & (period > 0)]
    if len(period_clean) == 0:
        return
    p_min = period_clean.min()
    p_max = period_clean.max()
    log_min = np.ceil(np.log2(p_min))
    log_max = np.floor(np.log2(p_max))
    if log_min >= log_max:
        log_min = np.floor(np.log2(p_min))
        log_max = np.ceil(np.log2(p_max))
    yticks = 2 ** np.arange(log_min, log_max + 1)
    ax.set_yticks(np.log2(yticks))
    ax.set_yticklabels([f"{p:.1f}" if p < 1 else f"{int(p)}" for p in yticks])
    ax.set_ylim(np.log2(p_min), np.log2(p_max))
    ax.invert_yaxis()


# ----------------------------------------------------------
# XWT plot
# ----------------------------------------------------------
def plot_xwt(col_x, col_y, scale_label):
    tag   = f"{col_x}_vs_{col_y}"
    print(f"  XWT: {col_x} <-> {col_y}")

    dat_x, t = prepare_series(col_x)
    dat_y, _ = prepare_series(col_y)
    n = min(len(dat_x), len(dat_y))
    dat_x, dat_y, t = dat_x[:n], dat_y[:n], t[:n]

    wave_x, scales, freqs, coi, _, _ = wavelet.cwt(dat_x, DT, DJ, S0, J, MOTHER)
    wave_y, _,      _,     _,  _, _ = wavelet.cwt(dat_y, DT, DJ, S0, J, MOTHER)

    xwt_power = wave_x * np.conj(wave_y)
    power     = np.abs(xwt_power)
    phase     = np.angle(xwt_power)
    period    = 1.0 / freqs

    signif_x, _ = wavelet.significance(1.0, DT, scales, 0, alpha=0,
                                        significance_level=1 - ALPHA,
                                        wavelet=MOTHER)
    signif_y, _ = wavelet.significance(1.0, DT, scales, 0, alpha=0,
                                        significance_level=1 - ALPHA,
                                        wavelet=MOTHER)
    sig95    = np.sqrt(signif_x[:, None] * signif_y[:, None])
    sig_mask = (power / sig95) > 1

    coi_plot = np.interp(t, np.linspace(t[0], t[-1], len(coi)), np.log2(coi))

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor("white")

    levels = np.percentile(power, np.linspace(0, 100, 50))
    cf = ax.contourf(t, np.log2(period), np.log2(power + 1e-10),
                     levels=np.log2(levels + 1e-10),
                     extend="both", cmap="RdYlBu_r")
    ax.contour(t, np.log2(period), sig_mask.astype(float),
               [0.5], colors="black", linewidths=1.5)
    ax.fill_between(t, coi_plot, np.log2(period).max(),
                    facecolor="white", alpha=0.4)
    ax.plot(t, coi_plot, "k--", linewidth=1, label="COI")

    draw_arrows(ax, t, period, phase, sig_mask, step=5)
    set_period_yaxis(ax, period)

    ax.set_xlim(t[0], t[-1])
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("Period (Years)", fontsize=11)
    ax.set_title(
        f"Cross Wavelet Transform (XWT)  —  {col_x} ↔ {col_y}  [{scale_label}]\n"
        f"Right → In-phase  |  Left → Anti-phase  |  "
        f"Up → {col_x} leads  |  Down → {col_y} leads",
        fontsize=11, fontweight="bold"
    )
    ax.legend(loc="lower right", fontsize=8)
    plt.colorbar(cf, ax=ax, label="log\u2082(XWT Power)", pad=0.01)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f"XWT_{tag}.png")
    plt.savefig(out, dpi=120)
    plt.close()
    print(f"    -> {out}")


# ----------------------------------------------------------
# WTC plot  (pycwt version-safe)
# ----------------------------------------------------------
def plot_wtc(col_x, col_y, scale_label):
    tag   = f"{col_x}_vs_{col_y}"
    print(f"  WTC: {col_x} <-> {col_y}")

    dat_x, t = prepare_series(col_x)
    dat_y, _ = prepare_series(col_y)
    n = min(len(dat_x), len(dat_y))
    dat_x, dat_y, t = dat_x[:n], dat_y[:n], t[:n]

    # pycwt version-safe unpacking
    import pycwt.helpers as _cwt_helpers

    # Monkey-patch ar1 to never raise Warning
    _orig_ar1 = _cwt_helpers.ar1
    def _safe_ar1(x):
        try:
            return _orig_ar1(x)
        except Warning:
            return 0.0   # white noise fallback
    _cwt_helpers.ar1 = _safe_ar1

    try:
        wct_result = wavelet.wct(
            dat_x, dat_y, DT, DJ, S0, J,
            significance_level=1 - ALPHA,
            wavelet=MOTHER, normalize=True, cache=True
        )
    except Exception as e:
        print(f"    [INFO] wct failed ({e}), skipping significance.")
        wct_result = None
    finally:
        _cwt_helpers.ar1 = _orig_ar1   # restore original

    if wct_result is None:
        return

    if len(wct_result) == 6:
        WTC, aWTC, _, coi, freqs, signif = wct_result
    elif len(wct_result) == 5:
        WTC, aWTC, _, coi, freqs = wct_result
        signif = np.ones(WTC.shape[0]) * 0.5
    else:
        raise ValueError(f"Unexpected number of return values: {len(wct_result)}")

    period = 1.0 / freqs
    # Remove invalid period values
    valid  = np.isfinite(period) & (period > 0)
    period = period[valid]
    WTC    = WTC[valid, :]
    aWTC   = aWTC[valid, :]
    if isinstance(signif, np.ndarray) and len(signif) == len(valid):
        signif = signif[valid]
    sig_mask = np.abs(WTC) > signif[:, None]
    coi_plot = np.interp(t, np.linspace(t[0], t[-1], len(coi)), np.log2(coi))

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor("white")

    cf = ax.contourf(t, np.log2(period), np.abs(WTC),
                     np.linspace(0, 1, 51), extend="both", cmap="RdYlGn")
    ax.contour(t, np.log2(period), sig_mask.astype(float),
               [0.5], colors="black", linewidths=1.5)
    ax.fill_between(t, coi_plot, np.log2(period).max(),
                    facecolor="white", alpha=0.4)
    ax.plot(t, coi_plot, "k--", linewidth=1, label="COI")

    draw_arrows(ax, t, period, aWTC, sig_mask, step=5)
    set_period_yaxis(ax, period)

    ax.set_xlim(t[0], t[-1])
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("Period (Years)", fontsize=11)
    ax.set_title(
        f"Wavelet Coherence (WTC)  —  {col_x} ↔ {col_y}  [{scale_label}]\n"
        f"Green = High Coherence (0\u21921)  |  Black contour = 95% significance  |  "
        f"Arrows = Phase relationship",
        fontsize=11, fontweight="bold"
    )
    ax.legend(loc="lower right", fontsize=8)
    cbar = plt.colorbar(cf, ax=ax, label="Coherence (0\u20131)", pad=0.01)
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f"WTC_{tag}.png")
    plt.savefig(out, dpi=120)
    plt.close()
    print(f"    -> {out}\n")


# ----------------------------------------------------------
# Main loop
# ----------------------------------------------------------
print("=" * 58)
print("  XWT & Wavelet Coherence Analysis — Starting")
print("=" * 58 + "\n")

for col_x, col_y, label in PAIRS:
    missing = [c for c in [col_x, col_y] if c not in df_raw.columns]
    if missing:
        print(f"[WARNING] Columns {missing} not found, skipping.\n")
        continue
    plot_xwt(col_x, col_y, label)
    plot_wtc(col_x, col_y, label)

print("\nAll XWT and WTC plots completed!")
print(f"Output folder: {os.path.abspath(OUTPUT_DIR)}")

print("""
╔══════════════════════════════════════════════════════════╗
║            INTERPRETATION GUIDE                          ║
╠══════════════════════════════════════════════════════════╣
║  XWT                                                     ║
║  • High power + inside black contour → both series       ║
║    oscillate strongly at the same period                 ║
║  • Right arrow  → in-phase (move together)               ║
║  • Left arrow   → anti-phase (move opposite)             ║
║  • Up arrow     → X leads Y                              ║
║  • Down arrow   → Y leads X                              ║
╠══════════════════════════════════════════════════════════╣
║  WTC                                                     ║
║  • Values 0→1: closer to 1 = high coherence             ║
║  • Dark green + black contour = significant coherence    ║
║  • Phase arrows same meaning as XWT                      ║
║  • White shaded area (COI) = unreliable edge region      ║
╚══════════════════════════════════════════════════════════╝
""")